# Notebook 12 — Daily Capital Allocation Advisor

**Goal**: Generate actionable daily trading suggestions.

---

## What This Advisor Does

Given a capital base (₹50K–1L) and the latest model predictions, this notebook:

1. **Loads** the latest ensemble predictions (or latest model predictions)
2. **Applies** position sizing via Half-Kelly criterion
3. **Runs** 6 risk checks (daily drawdown, concentration, sector, confidence, etc.)
4. **Outputs** clear BUY / SELL / HOLD / WAIT_FOR_DIP signals for each stock
5. **Formats** a human-readable daily suggestion table

### Signal Logic

| Signal | Condition |
|--------|----------|
| **STRONG_BUY** | Predicted return > 5%, confidence > 0.65 |
| **BUY** | Predicted return > 1%, confidence > 0.55 |
| **WAIT_FOR_DIP** | Bullish but above recent average price |
| **HOLD** | Moderate confidence, no strong edge |
| **SELL** | Predicted return < -1%, confidence > 0.55 |

In [1]:
import os, sys, gc, time
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

from src.portfolio.allocator import DailyAllocator
from src.portfolio.position_sizer import PositionSizer
from src.portfolio.risk_manager import RiskManager, PortfolioState
from src.utils.constants import *
from src.backtest.costs import GrowwCostCalculator

plt.style.use("seaborn-v0_8-whitegrid")

# Stabilize kernel after imports
gc.collect()
time.sleep(0.1)

print("✅ Setup complete")

✅ Setup complete


## Step 0 — Model Staleness Check

Before generating trading signals, verify that all models are fresh enough to trust.
This checks model age, accuracy drift, volatility regime shifts, and ticker coverage.

In [2]:
from src.utils.staleness import ModelStalenessDetector

staleness_detector = ModelStalenessDetector(models_dir=MODELS_DIR, max_age_days=30)

# Load price data for regime check
price_data_stale = {}
processed_dir_stale = DATA_DIR / "processed"
for f in sorted(processed_dir_stale.glob("*.parquet")):
    if f.name.endswith("_predictions.parquet") or f.name.startswith("ensemble"):
        continue
    try:
        df_tmp = pd.read_parquet(f)
        if "Close" in df_tmp.columns:
            ticker_name = f.stem
            price_data_stale[ticker_name] = df_tmp
    except Exception:
        pass

# Check all models
staleness_reports = staleness_detector.check_all_models(
    price_data=price_data_stale if price_data_stale else None,
)
staleness_text = staleness_detector.print_dashboard(staleness_reports)

# Warn if any model is STALE or EXPIRED
worst_status = max(r.n_flags for r in staleness_reports) if staleness_reports else 0
if worst_status >= 3:
    print("\n*** WARNING: Some models are STALE/EXPIRED. Consider retraining before trading! ***")
elif worst_status >= 1:
    print("\n*** NOTE: Some models are aging. Schedule retraining soon. ***")
else:
    print("\n*** All models are fresh. Proceeding with confidence. ***")

  MODEL STALENESS DASHBOARD
  Checked: 2026-02-11 14:08

  [OK] XGBoost          Status: FRESH     Age:    0d  Flags: 0/5
  [OK] LightGBM         Status: FRESH     Age:    0d  Flags: 0/5
  [OK] LSTM             Status: FRESH     Age:    0d  Flags: 0/5
  [OK] Transformer      Status: FRESH     Age:    0d  Flags: 0/5
  [OK] Ensemble         Status: FRESH     Age:    0d  Flags: 0/5

----------------------------------------------------------------------

  XGBoost
  ~~~~~~~
    Last trained:     2026-02-10 22:20
    Age:              0 days (max: 30)
    Age flag:         [OK]
    Training acc:     55.0%
    Recent acc:       0.0%
    Accuracy drop:    0.0%
    Accuracy flag:    [OK]
    Hist volatility:  26.42%
    Recent vol:       32.71%
    Vol ratio:        1.24
    Regime flag:      [OK]
    Coverage:         100%
    Coverage flag:    [OK]
    Action:           No action needed.


  LightGBM
  ~~~~~~~~
    Last trained:     2026-02-10 22:22
    Age:              0 days (max: 30)
   

## Step 1 — Configure the Advisor

Set your capital, max positions, risk limits, and the date you want suggestions for.

In [3]:
# ╔══════════════ YOUR CONFIGURATION ══════════════╗
CAPITAL = 50_000          # ₹50,000 starting capital
MAX_POSITIONS = 8         # Maximum concurrent positions
MAX_PER_POSITION = 0.12   # 12% max in any single stock
MIN_CONFIDENCE = 0.60     # Minimum model confidence to trade
RISK_FREE_RATE = 0.065    # 6.5% (Indian T-bill rate)
# ╚════════════════════════════════════════════════╝

print(f"Capital:              ₹{CAPITAL:>10,}")
print(f"Max positions:        {MAX_POSITIONS:>10}")
print(f"Max per position:     {MAX_PER_POSITION:>10.0%}")
print(f"Max per position (₹): ₹{CAPITAL * MAX_PER_POSITION:>10,.0f}")
print(f"Min confidence:       {MIN_CONFIDENCE:>10.0%}")

Capital:              ₹    50,000
Max positions:                 8
Max per position:            12%
Max per position (₹): ₹     6,000
Min confidence:              60%


## Step 2 — Load Latest Predictions

We load the most recent model predictions. The advisor works with any
model output that has columns: `ticker`, `predicted`, `actual`, `confidence` (optional).

In [4]:
# Generate real predictions per ticker using saved XGBoost model
import joblib
from src.features.pipeline import FeaturePipeline

predictions = None
model_used = "none"
current_prices = None

# Load price data
processed_dir = DATA_DIR / "processed"
price_files = sorted([f for f in processed_dir.glob("*.parquet") if "prediction" not in f.stem])
price_data = {}
for f in price_files:
    df = pd.read_parquet(f)
    if "Close" in df.columns:
        price_data[f.stem] = df

print(f"Loaded {len(price_data)} ticker datasets")

# Try to load XGBoost model and generate per-ticker predictions
xgb_model_path = MODELS_DIR / "xgboost_model.joblib"
if xgb_model_path.exists():
    from src.models import XGBoostModel
    model = XGBoostModel(task="regression")
    model.load(xgb_model_path)
    model_used = "xgboost"
    
    # Get the model's expected features
    try:
        model_features = model.model.feature_names_in_.tolist() if hasattr(model.model, 'feature_names_in_') else None
    except:
        model_features = None
    
    if model_features:
        print(f"Model expects {len(model_features)} features")
    
    pipeline = FeaturePipeline()
    
    # Predict using LAST N ROWS as panel (like training) for better differentiation
    all_latest = []
    failed = []
    for ticker, df in price_data.items():
        feature_cols = model_features if model_features else pipeline.get_feature_columns(df)
        
        # Check which features exist in df
        available = [c for c in feature_cols if c in df.columns]
        
        if len(available) < 10:
            failed.append((ticker, f"only {len(available)} features"))
            continue
        
        valid = df.dropna(subset=available)
        if len(valid) < 50:
            failed.append((ticker, f"only {len(valid)} valid rows"))
            continue
        
        # Use last 5 rows for each ticker — average prediction for stability
        last_rows = valid.iloc[-5:]
        try:
            # Build feature matrix — use NaN for missing features (XGBoost handles NaN natively)
            X = pd.DataFrame(np.nan, index=last_rows.index, columns=feature_cols)
            for c in available:
                X[c] = last_rows[c].values
            preds = model.predict(X)
            pred_val = float(np.mean(preds))
        except Exception as e:
            failed.append((ticker, str(e)[:80]))
            continue
        
        all_latest.append({
            "Ticker": ticker,
            "raw_pred": pred_val,
        })
    
    if failed:
        print(f"  ⚠ {len(failed)} tickers failed (showing first 3):")
        for t, reason in failed[:3]:
            print(f"    {t}: {reason}")
    
    if len(all_latest) > 0:
        raw_df = pd.DataFrame(all_latest)
        
        # RANK-BASED signal construction (how real quant strategies work)
        # Top 25% → BUY candidates, Bottom 25% → SELL, Middle → HOLD
        raw_df["pred_rank"] = raw_df["raw_pred"].rank(pct=True)
        
        rows = []
        for _, row in raw_df.iterrows():
            pred_val = row["raw_pred"]
            rank_pct = row["pred_rank"]
            
            # Direction_Prob based on percentile rank (top performers get high prob)
            direction_prob = 0.40 + rank_pct * 0.50  # Range: 0.40 to 0.90
            
            # Confidence based on how extreme the rank is (top/bottom = high confidence)
            extremity = abs(rank_pct - 0.5) * 2  # 0 at median, 1 at extremes
            confidence = 0.45 + extremity * 0.45  # Range: 0.45 to 0.90
            
            rows.append({
                "Ticker": row["Ticker"],
                "Predicted_Range": abs(pred_val),
                "Direction_Prob": direction_prob,
                "Confidence": confidence,
                "Raw_Prediction": pred_val,
                "Rank_Percentile": rank_pct,
            })
        
        predictions = pd.DataFrame(rows)
        current_prices = pd.Series({t: price_data[t]["Close"].iloc[-1] for t in price_data})
        
        n_buy = (predictions["Direction_Prob"] > 0.58).sum()
        n_strong = (predictions["Direction_Prob"] > 0.70).sum()
        print(f"✅ Generated {len(predictions)} per-ticker predictions using XGBoost model")
        print(f"   Rank-based signals: {n_strong} STRONG_BUY, {n_buy - n_strong} BUY candidates")
        print(f"   Prediction range: [{raw_df['raw_pred'].min():.6f}, {raw_df['raw_pred'].max():.6f}]")
    else:
        print(f"⚠ No predictions generated. All {len(price_data)} tickers failed.")
    
if predictions is None or len(predictions) == 0:
    print("⚠️ Falling back to demo predictions")
    model_used = "demo"
    import yaml
    with open(str(PROJECT_ROOT) + "/config/tickers.yaml") as f:
        tk = yaml.safe_load(f)
    demo_tickers = []
    for bucket in tk["buckets"].values():
        demo_tickers.extend(bucket[:5])
    
    np.random.seed(42)
    predictions = pd.DataFrame({
        "Ticker": demo_tickers,
        "Predicted_Range": np.random.uniform(0.01, 0.08, len(demo_tickers)),
        "Direction_Prob": np.random.uniform(0.40, 0.85, len(demo_tickers)),
        "Confidence": np.random.uniform(0.45, 0.85, len(demo_tickers)),
    })
    current_prices = pd.Series(
        np.random.uniform(100, 5000, len(demo_tickers)), index=demo_tickers,
    )

print(f"\nModel: {model_used}")
print(f"Predictions: {len(predictions)} tickers")
predictions.head(10)

Loaded 122 ticker datasets
  Loaded xgboost from /Users/anto/Trading_Project/masters_trading_ai/models/xgboost_model.joblib
Model expects 206 features
✅ Generated 122 per-ticker predictions using XGBoost model
   Rank-based signals: 0 STRONG_BUY, 122 BUY candidates
   Prediction range: [0.005405, 0.005405]

Model: xgboost
Predictions: 122 tickers


,Ticker,Predicted_Range,Direction_Prob,Confidence,Raw_Prediction,Rank_Percentile
0,ABB.NS,0.005405,0.652049,0.453689,0.005405,0.504098
1,ADANIENT.NS,0.005405,0.652049,0.453689,0.005405,0.504098
2,ADANIGREEN.NS,0.005405,0.652049,0.453689,0.005405,0.504098
3,ADANIPORTS.NS,0.005405,0.652049,0.453689,0.005405,0.504098
4,ADANIPOWER.NS,0.005405,0.652049,0.453689,0.005405,0.504098
5,AMBUJACEM.NS,0.005405,0.652049,0.453689,0.005405,0.504098
6,APOLLOHOSP.NS,0.005405,0.652049,0.453689,0.005405,0.504098
7,ASIANPAINT.NS,0.005405,0.652049,0.453689,0.005405,0.504098
8,ASTRAL.NS,0.005405,0.652049,0.453689,0.005405,0.504098
9,AUBANK.NS,0.005405,0.652049,0.453689,0.005405,0.504098


## Step 3 — Initialize Components

Three components work together:

1. **PositionSizer** — calculates how much to allocate using Half-Kelly
2. **RiskManager** — validates each trade against 6 risk constraints
3. **DailyAllocator** — combines everything into actionable signals

In [5]:
# Initialize components
# PositionSizer only accepts method= (not total_capital, max_position_pct, etc.)
sizer = PositionSizer(method="half_kelly")

# RiskManager only accepts config_path= (reads limits from settings.yaml)
risk_manager = RiskManager()

# DailyAllocator uses capital= (not total_capital=)
allocator = DailyAllocator(
    capital=CAPITAL,
    max_positions=MAX_POSITIONS,
    max_position_pct=MAX_PER_POSITION,
    min_confidence=MIN_CONFIDENCE,
)

cost_calc = GrowwCostCalculator()

print("✅ Advisor components initialized")
print(f"   • PositionSizer: Half-Kelly, max {MAX_PER_POSITION:.0%}/position")
print(f"   • RiskManager: 6 risk checks active")
print(f"   • DailyAllocator: Edge-weighted allocation")
print(f"   • GrowwCostCalculator: Real fee deduction")

✅ Advisor components initialized
   • PositionSizer: Half-Kelly, max 12%/position
   • RiskManager: 6 risk checks active
   • DailyAllocator: Edge-weighted allocation
   • GrowwCostCalculator: Real fee deduction


## Step 4 — Generate Daily Suggestions

This is the **core output** of the trading advisor. For each stock:

1. Check model prediction & confidence
2. Compute position size via Half-Kelly
3. Run risk checks (drawdown, concentration, etc.)
4. Generate **BUY / SELL / HOLD / WAIT_FOR_DIP** signal
5. Estimate Groww transaction costs

In [6]:
# Generate allocations
# DailyAllocator.allocate() requires (predictions, current_prices)
# — NOT generate_allocations(latest)
if "Ticker" in predictions.columns:
    latest = predictions
elif "Ticker" not in predictions.columns and model_used != "demo":
    latest = predictions.tail(25)  # Last day's predictions
else:
    latest = predictions

allocations = allocator.allocate(latest, current_prices)

# format_suggestions() returns a DataFrame — use display()
suggestion_df = allocator.format_suggestions(allocations)
display(suggestion_df)

,Action,Ticker,Amount (₹),Shares,Price,Pred. Direction,Pred. Range,Confidence,Bucket,Rationale
0,WAIT_FOR_DIP,POWERGRID.NS,"₹4,725",15,₹314.98,65%,0.54%,45%,unknown,"Wait for better entry — range 0.54%, direction..."
1,WAIT_FOR_DIP,PNB.NS,"₹4,695",47,₹99.89,65%,0.54%,45%,unknown,"Wait for better entry — range 0.54%, direction..."
2,WAIT_FOR_DIP,PIIND.NS,"₹3,116",1,"₹3,115.88",65%,0.54%,45%,unknown,"Wait for better entry — range 0.54%, direction..."
3,WAIT_FOR_DIP,PIDILITIND.NS,"₹3,218",2,"₹1,608.86",65%,0.54%,45%,unknown,"Wait for better entry — range 0.54%, direction..."
4,WAIT_FOR_DIP,PFC.NS,"₹4,671",10,₹467.13,65%,0.54%,45%,unknown,"Wait for better entry — range 0.54%, direction..."
5,WAIT_FOR_DIP,PAYTM.NS,"₹3,926",3,"₹1,308.73",65%,0.54%,45%,unknown,"Wait for better entry — range 0.54%, direction..."


## Step 5 — Risk Check Each Allocation

Before executing any trade, the RiskManager validates against 6 constraints:

1. **Daily drawdown** < 3%
2. **Total drawdown** < 15%
3. **Position concentration** < 15%
4. **Sector concentration** < 35%
5. **Concurrent positions** < 10
6. **Minimum confidence** > 55%

In [7]:
# Simulate current portfolio state
# PortfolioState fields: capital, peak_capital, daily_start, positions, sector_map
# — NOT current_capital, daily_start_capital, open_positions, daily_pnl
portfolio_state = PortfolioState(
    capital=CAPITAL,
    peak_capital=CAPITAL,
    daily_start=CAPITAL,
    positions={},
    sector_map={},
)

# Run risk checks on each allocation
approved = []
rejected = []

for alloc in allocations:
    if alloc.action in ["BUY", "STRONG_BUY"]:
        # approve_trade() signature: (ticker, amount, confidence, state)
        # Returns List[RiskCheck], each with .passed and .reason
        checks = risk_manager.approve_trade(
            ticker=alloc.ticker,
            amount=alloc.amount_inr,
            confidence=alloc.confidence,
            state=portfolio_state,
        )
        if all(c.passed for c in checks):
            approved.append(alloc)
        else:
            failed = [c for c in checks if not c.passed]
            rejected.append((alloc, "; ".join(c.reason for c in failed)))

print(f"\n✅ APPROVED TRADES: {len(approved)}")
print(f"❌ REJECTED TRADES: {len(rejected)}")
print()

# Show approved
if approved:
    print("╔══ APPROVED TRADES ══╗")
    for a in approved:
        # GrowwCostCalculator has buy_cost() — NOT calculate_costs()
        # TradeCost property is .total — NOT .total_cost
        cost = cost_calc.buy_cost(a.amount_inr, "equity_delivery")
        print(f"  {a.ticker:15s} | {a.action:10s} | ₹{a.amount_inr:>8,.0f} | "
              f"Confidence: {a.confidence:.0%} | Groww Cost: ₹{cost.total:.2f}")

# Show rejected with reasons
if rejected:
    print("\n╔══ REJECTED BY RISK MANAGER ══╗")
    for a, reason in rejected:
        print(f"  {a.ticker:15s} | {a.action:10s} | Reason: {reason}")


✅ APPROVED TRADES: 0
❌ REJECTED TRADES: 0



## Step 6 — Position Sizing Details

Half-Kelly Formula:

$$f^* = \frac{1}{2} \times \frac{p \cdot b - q}{b}$$

Where:
- $p$ = win probability (from model confidence)
- $q = 1 - p$ = loss probability  
- $b$ = win/loss ratio
- Factor of $\frac{1}{2}$ = Half-Kelly (conservative)

In [8]:
# Show position sizing for top picks
# Allocation has no .edge attribute — compute from its fields
buy_allocs = [a for a in allocations if a.action in ["BUY", "STRONG_BUY"]]
buy_allocs.sort(
    key=lambda x: x.predicted_range_pct * x.predicted_direction * x.confidence,
    reverse=True,
)

if buy_allocs:
    print("Position Sizing Details (Half-Kelly)")
    print("=" * 75)
    print(f"{'Ticker':15s} | {'Edge':>8s} | {'Kelly %':>8s} | {'Amount':>10s} | {'% of Capital':>12s}")
    print("-" * 75)
    
    total_deployed = 0
    for a in buy_allocs[:MAX_POSITIONS]:
        edge = a.predicted_range_pct * a.predicted_direction * a.confidence

        # kelly_fraction() signature: (win_prob, avg_win, avg_loss)
        # — NOT (win_prob, win_loss_ratio)
        avg_win = max(a.predicted_range_pct, 0.01)
        avg_loss = 0.02  # Assume 2% average loss
        kelly_pct = sizer.kelly_fraction(
            win_prob=a.predicted_direction,
            avg_win=avg_win,
            avg_loss=avg_loss,
        )
        # Allocation uses .amount_inr — NOT .amount
        amount = min(a.amount_inr, CAPITAL * MAX_PER_POSITION)
        pct = amount / CAPITAL
        total_deployed += amount
        
        print(f"{a.ticker:15s} | {edge:>7.2%} | {kelly_pct:>7.2%} | "
              f"₹{amount:>9,.0f} | {pct:>11.1%}")
    
    print("-" * 75)
    print(f"{'TOTAL':15s} | {'':>8s} | {'':>8s} | "
          f"₹{total_deployed:>9,.0f} | {total_deployed/CAPITAL:>11.1%}")
    print(f"{'CASH RESERVE':15s} | {'':>8s} | {'':>8s} | "
          f"₹{CAPITAL - total_deployed:>9,.0f} | {1 - total_deployed/CAPITAL:>11.1%}")

## Step 7 — Transaction Cost Estimate

Estimate total Groww fees for executing all approved trades.

In [9]:
if approved:
    total_cost = 0
    cost_breakdown = []
    
    for a in approved:
        # buy_cost() not calculate_costs(); .total not .total_cost; .amount_inr not .amount
        cost = cost_calc.buy_cost(a.amount_inr, "equity_delivery")
        total_cost += cost.total
        cost_breakdown.append({
            "Ticker": a.ticker,
            "Trade Amount": a.amount_inr,
            "Brokerage": cost.brokerage,
            "STT": cost.stt,
            "GST": cost.gst,
            "Stamp Duty": cost.stamp_duty,
            "Total Cost": cost.total,
            "Cost %": cost.total / a.amount_inr * 100
        })
    
    df_costs = pd.DataFrame(cost_breakdown)
    
    print(f"Total Transaction Costs: ₹{total_cost:,.2f}")
    print(f"Cost as % of deployed capital: {total_cost / sum(a.amount_inr for a in approved) * 100:.3f}%")
    print()
    display(df_costs.round(2))

## Step 8 — Capital Allocation Visualization

In [10]:
if buy_allocs:
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    
    # 1. Allocation pie chart
    top_allocs = buy_allocs[:MAX_POSITIONS]
    # .amount_inr not .amount
    amounts = [a.amount_inr for a in top_allocs]
    labels = [a.ticker.replace(".NS", "") for a in top_allocs]
    cash = CAPITAL - sum(amounts)
    if cash > 0:
        amounts.append(cash)
        labels.append("CASH")
    
    colors = plt.cm.Set3(np.linspace(0, 1, len(amounts)))
    wedges, texts, autotexts = axes[0].pie(
        amounts, labels=labels, autopct="%1.1f%%",
        colors=colors, pctdistance=0.85,
        wedgeprops={"edgecolor": "white", "linewidth": 1}
    )
    axes[0].set_title(f"Capital Allocation (₹{CAPITAL:,})")
    
    # 2. Confidence vs Edge scatter
    # Allocation has no .edge — compute from fields
    for a in allocations:
        edge = a.predicted_range_pct * a.predicted_direction * a.confidence
        color = {"STRONG_BUY": "green", "BUY": "limegreen",
                 "HOLD": "gold", "SELL": "red",
                 "WAIT_FOR_DIP": "orange"}.get(a.action, "gray")
        axes[1].scatter(a.confidence, edge * 100,
                        c=color, s=100, alpha=0.7, edgecolors="black", lw=0.5)
        axes[1].annotate(a.ticker.replace(".NS", ""),
                          (a.confidence, edge * 100),
                          fontsize=7, ha="center", va="bottom")
    
    axes[1].axhline(0, color="black", linestyle="--", alpha=0.3)
    axes[1].axvline(MIN_CONFIDENCE, color="red", linestyle="--",
                     alpha=0.5, label=f"Min Conf = {MIN_CONFIDENCE:.0%}")
    axes[1].set_xlabel("Model Confidence")
    axes[1].set_ylabel("Expected Edge (%)")
    axes[1].set_title("Confidence vs Edge (Action Map)")
    axes[1].legend()
    
    plt.tight_layout()
    plt.savefig(REPORTS_DIR / "daily_allocation.png", dpi=150, bbox_inches="tight")
    plt.show()

## Step 9 — Risk Report

In [11]:
report = risk_manager.format_risk_report(portfolio_state)
print(report)

╔══════════════ Risk Dashboard ══════════════╗
  Capital:   ₹      50,000
  Peak:      ₹      50,000
  Daily DD:    0.00%  (limit 3.00%)
  Total DD:    0.00%  (limit 15.00%)
  Positions:    0       (limit 15)
  Cash:      100.00%
╚════════════════════════════════════════════╝


## Step 10 — Technical Entry/Exit Levels (RSI, MACD, Bollinger, SMA)

For each recommended ticker, compute real-time technical indicator levels to guide
intraday entry/exit decisions between 9:15 AM and 3:30 PM IST.

In [12]:
# Compute technical entry/exit levels for recommended tickers
from src.utils.report_generator import compute_technical_levels

# Get tickers with BUY or STRONG_BUY signals
recommended_tickers = [a.ticker for a in allocations if a.action in ("BUY", "STRONG_BUY")]
# If none, show top watchlist tickers
if not recommended_tickers:
    recommended_tickers = [a.ticker for a in allocations if a.action == "WAIT_FOR_DIP"][:10]

if recommended_tickers and price_data:
    tech_levels = compute_technical_levels(price_data, tickers=recommended_tickers)
    
    if len(tech_levels) > 0:
        print("╔══════════════ TECHNICAL ENTRY/EXIT LEVELS ══════════════╗")
        print(f"║  Trading Window: 9:15 AM — 3:30 PM IST                  ║")
        print(f"║  Tickers shown: {len(tech_levels):>3}                                      ║")
        print("╠══════════════════════════════════════════════════════════╣\n")
        
        # Signal interpretation guide
        print("Signal Guide:")
        print("  ENTRY  → RSI < 40 + MACD histogram positive (oversold + momentum turning)")
        print("  EXIT   → RSI > 70 or MACD histogram negative (overbought or momentum fading)")
        print("  NEUTRAL → No clear signal — wait for confirmation\n")
        
        display(tech_levels)
        
        # Detailed per-ticker analysis
        print("\n📊 Per-Ticker Technical Summary:")
        for _, row in tech_levels.iterrows():
            print(f"\n  {row['Ticker']:12s} | Price: {row['Price']} | RSI: {row['RSI']} | "
                  f"MACD_H: {row['MACD_Hist']} | BB: {row['BB_Pos']}")
            print(f"  {'':12s} | Entry: {row['Entry']} | StopLoss: {row['StopLoss']} | "
                  f"Target: {row['Target']} | Signal: {row['Signal']}")
    else:
        print("Technical data not available for recommended tickers")
else:
    print("No recommended tickers or price data available")

╔══════════════ TECHNICAL ENTRY/EXIT LEVELS ══════════════╗
║  Trading Window: 9:15 AM — 3:30 PM IST                  ║
║  Tickers shown:   6                                      ║
╠══════════════════════════════════════════════════════════╣

Signal Guide:
  ENTRY  → RSI < 40 + MACD histogram positive (oversold + momentum turning)
  EXIT   → RSI > 70 or MACD histogram negative (overbought or momentum fading)
  NEUTRAL → No clear signal — wait for confirmation



,Ticker,Price,RSI,MACD_Hist,BB_Pos,SMA20,SMA50,Entry,StopLoss,Target,Signal
0,PAYTM,"Rs.1,308.73",41.1,-0.43,0.32,"Rs.1,362","Rs.1,420","Rs.1,306.11",Rs.967.09,"Rs.1,821.18",EXIT
1,PFC,Rs.467.13,70.3,5.39,0.92,Rs.429,Rs.410,Rs.466.20,Rs.348.11,Rs.645.66,EXIT
2,PIDILITIND,"Rs.1,608.86",57.3,5.31,0.74,"Rs.1,579","Rs.1,586","Rs.1,605.64","Rs.1,322.50","Rs.2,038.40",NEUTRAL
3,PIIND,"Rs.3,115.88",51.2,4.92,0.60,"Rs.3,092","Rs.3,147","Rs.3,109.64","Rs.2,780.94","Rs.3,618.29",NEUTRAL
4,PNB,Rs.99.89,48.9,-0.26,0.37,Rs.101,Rs.99,Rs.99.69,Rs.47.61,Rs.178.31,EXIT
5,POWERGRID,Rs.314.98,75.6,4.70,0.97,Rs.286,Rs.285,Rs.314.35,Rs.257.12,Rs.401.76,EXIT



📊 Per-Ticker Technical Summary:

  PAYTM        | Price: Rs.1,308.73 | RSI: 41.1 | MACD_H: -0.43 | BB: 0.32
               | Entry: Rs.1,306.11 | StopLoss: Rs.967.09 | Target: Rs.1,821.18 | Signal: EXIT

  PFC          | Price: Rs.467.13 | RSI: 70.3 | MACD_H: 5.39 | BB: 0.92
               | Entry: Rs.466.20 | StopLoss: Rs.348.11 | Target: Rs.645.66 | Signal: EXIT

  PIDILITIND   | Price: Rs.1,608.86 | RSI: 57.3 | MACD_H: 5.31 | BB: 0.74
               | Entry: Rs.1,605.64 | StopLoss: Rs.1,322.50 | Target: Rs.2,038.40 | Signal: NEUTRAL

  PIIND        | Price: Rs.3,115.88 | RSI: 51.2 | MACD_H: 4.92 | BB: 0.60
               | Entry: Rs.3,109.64 | StopLoss: Rs.2,780.94 | Target: Rs.3,618.29 | Signal: NEUTRAL

  PNB          | Price: Rs.99.89 | RSI: 48.9 | MACD_H: -0.26 | BB: 0.37
               | Entry: Rs.99.69 | StopLoss: Rs.47.61 | Target: Rs.178.31 | Signal: EXIT

  POWERGRID    | Price: Rs.314.98 | RSI: 75.6 | MACD_H: 4.70 | BB: 0.97
               | Entry: Rs.314.35 | StopLoss: R

## Step 11 — Generate PDF Daily Strategy Report

Creates a professional PDF report with all trading signals, technical levels,
risk metrics, and transaction cost estimates.

In [14]:
import importlib
import src.utils.report_generator as _rg
importlib.reload(_rg)
from src.utils.report_generator import DailyReportGenerator

# Build model metrics dict from available predictions
model_metrics = {}
for name in ["xgboost", "lightgbm", "lstm", "transformer", "ensemble"]:
    p = DATA_DIR / "processed" / f"{name}_predictions.parquet"
    if p.exists():
        df_pred = pd.read_parquet(p)
        rmse = np.sqrt(np.mean((df_pred["actual"] - df_pred["predicted"])**2))
        dir_acc = np.mean(np.sign(df_pred["predicted"]) == np.sign(df_pred["actual"]))
        from scipy.stats import spearmanr
        ic, _ = spearmanr(df_pred["predicted"], df_pred["actual"])
        model_metrics[name] = {"rmse": rmse, "dir_acc": dir_acc, "ic": ic, "active": True}

# Build risk metrics dict
total_deployed = sum(a.amount_inr for a in allocations if a.action in ('BUY', 'STRONG_BUY'))
risk_metrics_dict = {
    "Max Position Size": {"value": f"{MAX_PER_POSITION:.0%}", "limit": "15%", "ok": True},
    "Max Positions": {"value": str(MAX_POSITIONS), "limit": "8", "ok": True},
    "Capital Deployed": {
        "value": f"Rs.{total_deployed:,.0f}",
        "limit": f"Rs.{CAPITAL:,.0f}",
        "ok": True,
    },
}

# Technical levels
tech_df = tech_levels if 'tech_levels' in dir() and len(tech_levels) > 0 else pd.DataFrame()

# Generate PDF
report_gen = DailyReportGenerator(capital=CAPITAL, reports_dir=str(REPORTS_DIR), broker="Groww")
pdf_path = report_gen.generate(
    allocations=allocations,
    predictions=predictions,
    price_data=price_data,
    model_metrics=model_metrics,
    risk_metrics=risk_metrics_dict,
    technical_levels=tech_df,
)
print(f"\nOpen the PDF: {pdf_path}")

[OK] Report saved: /Users/anto/Trading_Project/masters_trading_ai/reports/daily_strategy_2026-02-11.pdf

Open the PDF: /Users/anto/Trading_Project/masters_trading_ai/reports/daily_strategy_2026-02-11.pdf


## ✅ Daily Advisor Complete!

**What you got:**
- Clear BUY / SELL / HOLD / WAIT_FOR_DIP signals
- Half-Kelly position sizing
- Risk-checked allocations (6 constraints)
- Groww transaction cost estimates
- Visual capital allocation

**Next:** Proceed to **Notebook 13 — Options Lab (Nifty/BankNifty)**

---
*Educational project. Not financial advice.*

---
## Interactive Single-Stock Deep Analyzer

Enter **ANY** stock ticker (NSE/BSE/US) and get a complete institutional-grade analysis:
- 50+ technical indicators with multi-factor scoring
- Directional signal: **BUY / SELL / SHORT / LONG / HOLD** with confidence
- ATR-based entry, stop-loss, and 3 profit targets
- Full options Greeks: **Delta, Gamma, Theta, Vega, Rho**
- Options strategy recommendation (spreads, straddle, iron condor)
- Portfolio risk metrics: **Alpha, Beta, Sharpe, Sortino, VaR, Max Drawdown**
- Support/Resistance with pivot points

### Ticker Format
| Market | Format | Example |
|--------|--------|---------|
| NSE India | `TICKER.NS` | `RELIANCE.NS`, `TCS.NS`, `INFY.NS` |
| BSE India | `TICKER.BO` | `RELIANCE.BO` |
| US Stocks | `TICKER` | `AAPL`, `TSLA`, `GOOGL` |

In [15]:
# ============================================================
# Interactive Stock Analyzer — Enter ANY Ticker
# ============================================================
import importlib
import src.utils.stock_analyzer as _sa_mod
importlib.reload(_sa_mod)
from src.utils.stock_analyzer import StockAnalyzer

# Initialize analyzer with your capital
analyzer = StockAnalyzer(capital=CAPITAL, risk_free_rate=RISK_FREE_RATE)

# ╔══════════════════════════════════════════════════════════╗
# ║  CHANGE THE TICKER BELOW TO ANALYZE ANY STOCK           ║
# ║  NSE: "RELIANCE.NS", "TCS.NS", "INFY.NS", "SBIN.NS"   ║
# ║  BSE: "RELIANCE.BO"                                     ║
# ║  US:  "AAPL", "TSLA", "GOOGL", "MSFT"                  ║
# ╚══════════════════════════════════════════════════════════╝

TICKER = "RELIANCE.NS"   # <-- ENTER YOUR TICKER HERE

# Run full analysis
result = analyzer.analyze(TICKER)
analyzer.print_report(result)

  496 candles loaded (2024-02-12 to 2026-02-11)
Computing technical indicators...
Generating trading signal...
Analyzing options & Greeks...
Computing risk metrics (vs Nifty 50)...
  DEEP STOCK ANALYSIS: RELIANCE.NS
  Reliance Industries Limited | Energy | MCap: Rs.19.9T
  Generated: 2026-02-11 20:54:32

  !! PRIMARY SIGNAL: SHORT !!
  Conviction: MEDIUM | Score: 40/100
  Rationale: No strong conviction signals

  PRICE & TREND---------------------------------------------------------
  Price: Rs.1,468.70  |  1D: +0.70%  |  5D: +0.82%  |  20D: +1.09%
  SMA20: 1,425.4  |  SMA50: 1,493.8  |  SMA200: 1,448.4
  Trend: SIDEWAYS  |  ADX: 35.4 (STRONG_TREND)

  MOMENTUM--------------------------------------------------------------
  RSI(14): 69.2 [NEUTRAL]  |  Stoch K/D: 86/83
  MACD: -8.85 (Signal: -20.25)  |  Hist: +11.40  |  Cross: NONE
  CCI: 77  |  Williams%%R: -14

  VOLATILITY------------------------------------------------------------
  ATR(14): Rs.29.96 (2.04%)  |  Regime: NORMAL
  HV

In [16]:
# ============================================================
# Visualization: Price Chart with Technical Levels
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Reload data for charts
chart_df = analyzer.download_data(TICKER, period="6mo")
close = chart_df["Close"]
t = result.technical
s = result.signal

fig, axes = plt.subplots(4, 1, figsize=(14, 16), gridspec_kw={"height_ratios": [3, 1, 1, 1]})
fig.suptitle(f"Deep Analysis: {result.company_name} ({TICKER})", fontsize=16, fontweight="bold")

# --- Panel 1: Price + Bollinger + S/R ---
ax1 = axes[0]
ax1.plot(close.index, close, color="#1a1a2e", linewidth=1.5, label="Close")
sma20 = close.rolling(20).mean()
sma50 = close.rolling(50).mean()
ax1.plot(close.index, sma20, color="#e94560", linewidth=1, alpha=0.8, label="SMA 20")
ax1.plot(close.index, sma50, color="#0f3460", linewidth=1, alpha=0.8, label="SMA 50")

# Bollinger Bands
bb_mid = sma20
bb_std = close.rolling(20).std()
bb_up = bb_mid + 2 * bb_std
bb_lo = bb_mid - 2 * bb_std
ax1.fill_between(close.index, bb_lo, bb_up, alpha=0.1, color="#e94560", label="BB(20,2)")

# Entry/Exit lines
ax1.axhline(s.entry_price, color="green", ls="--", lw=1.2, alpha=0.7, label=f"Entry {s.entry_price:,.0f}")
ax1.axhline(s.stop_loss, color="red", ls="--", lw=1.2, alpha=0.7, label=f"Stop {s.stop_loss:,.0f}")
ax1.axhline(s.target_1, color="blue", ls=":", lw=1, alpha=0.6, label=f"T1 {s.target_1:,.0f}")
ax1.axhline(s.target_2, color="blue", ls="-.", lw=1, alpha=0.6, label=f"T2 {s.target_2:,.0f}")

# Support/Resistance
ax1.axhline(t.support_1, color="orange", ls="--", lw=0.8, alpha=0.5)
ax1.axhline(t.resistance_1, color="purple", ls="--", lw=0.8, alpha=0.5)
ax1.annotate(f"S1={t.support_1:,.0f}", xy=(close.index[-1], t.support_1),
             fontsize=7, color="orange", ha="right")
ax1.annotate(f"R1={t.resistance_1:,.0f}", xy=(close.index[-1], t.resistance_1),
             fontsize=7, color="purple", ha="right")

# Signal badge
badge_color = {"STRONG_BUY": "green", "BUY": "limegreen", "LONG": "lightgreen",
               "STRONG_SELL": "red", "SHORT": "salmon", "SELL": "lightsalmon",
               "HOLD": "gold"}.get(s.primary_action, "gray")
ax1.annotate(f" {s.primary_action} ({s.confidence_score}/100)",
             xy=(0.02, 0.95), xycoords="axes fraction", fontsize=12, fontweight="bold",
             color="white", bbox=dict(boxstyle="round,pad=0.3", fc=badge_color, alpha=0.9))

ax1.set_title("Price Action + Bollinger Bands + Entry/Exit Levels")
ax1.legend(loc="upper right", fontsize=7, ncol=4)
ax1.grid(True, alpha=0.3)

# --- Panel 2: RSI ---
ax2 = axes[1]
delta_c = close.diff()
gain = delta_c.clip(lower=0).rolling(14).mean()
loss = (-delta_c.clip(upper=0)).rolling(14).mean()
rs = gain / loss.replace(0, 1e-10)
rsi = 100 - (100 / (1 + rs))
ax2.plot(rsi.index, rsi, color="#e94560", linewidth=1.2)
ax2.axhline(70, color="red", ls="--", lw=0.8, alpha=0.6)
ax2.axhline(30, color="green", ls="--", lw=0.8, alpha=0.6)
ax2.fill_between(rsi.index, 30, 70, alpha=0.05, color="gray")
ax2.set_title(f"RSI(14) = {t.rsi_14:.1f} [{t.rsi_signal}]", fontsize=10)
ax2.set_ylim(0, 100)
ax2.grid(True, alpha=0.3)

# --- Panel 3: MACD ---
ax3 = axes[2]
ema12 = close.ewm(span=12).mean()
ema26 = close.ewm(span=26).mean()
macd_line = ema12 - ema26
macd_sig = macd_line.ewm(span=9).mean()
macd_hist = macd_line - macd_sig
colors_macd = ["green" if v >= 0 else "red" for v in macd_hist]
ax3.bar(macd_hist.index, macd_hist, color=colors_macd, alpha=0.6, width=1)
ax3.plot(macd_line.index, macd_line, color="#1a1a2e", linewidth=1, label="MACD")
ax3.plot(macd_sig.index, macd_sig, color="#e94560", linewidth=1, label="Signal")
ax3.axhline(0, color="gray", ls="-", lw=0.5)
ax3.set_title(f"MACD [{t.macd_cross}]", fontsize=10)
ax3.legend(loc="upper right", fontsize=7)
ax3.grid(True, alpha=0.3)

# --- Panel 4: Volume ---
ax4 = axes[3]
vol = chart_df["Volume"]
vol_sma = vol.rolling(20).mean()
colors_vol = ["green" if close.diff().iloc[i] > 0 else "red" for i in range(len(close))]
ax4.bar(vol.index, vol, color=colors_vol, alpha=0.5, width=1)
ax4.plot(vol_sma.index, vol_sma, color="#0f3460", linewidth=1.2, label="Vol SMA(20)")
ax4.set_title(f"Volume (Ratio: {t.volume_ratio:.2f}x | OBV: {t.obv_trend})", fontsize=10)
ax4.legend(loc="upper right", fontsize=7)
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(REPORTS_DIR / f"deep_analysis_{TICKER.replace('.', '_')}.png"), dpi=150, bbox_inches="tight")
plt.show()
print(f"\nChart saved to reports/deep_analysis_{TICKER.replace('.', '_')}.png")


Chart saved to reports/deep_analysis_RELIANCE_NS.png


In [17]:
# ============================================================
# Greeks Surface + Risk Dashboard
# ============================================================
import numpy as np
from src.options.greeks import BlackScholesGreeks

o = result.options
r = result.risk
bs = BlackScholesGreeks(RISK_FREE_RATE)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle(f"Options Greeks & Risk Dashboard: {TICKER}", fontsize=14, fontweight="bold")

S = o.spot
sigma = o.hist_vol
T = 30 / 365.0

# Strike range for Greek curves
strikes = np.linspace(S * 0.85, S * 1.15, 50)

# Panel 1: Delta vs Strike
ax = axes[0, 0]
deltas_c = [bs.delta(S, K, T, sigma, "call") for K in strikes]
deltas_p = [bs.delta(S, K, T, sigma, "put") for K in strikes]
ax.plot(strikes, deltas_c, "g-", lw=2, label="Call Delta")
ax.plot(strikes, deltas_p, "r-", lw=2, label="Put Delta")
ax.axvline(S, color="gray", ls="--", lw=0.8, label=f"Spot={S:,.0f}")
ax.axvline(o.atm_strike, color="blue", ls=":", lw=0.8, label=f"ATM={o.atm_strike:,.0f}")
ax.set_title(f"Delta = {o.delta:+.4f}", fontweight="bold")
ax.set_xlabel("Strike")
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

# Panel 2: Gamma vs Strike
ax = axes[0, 1]
gammas = [bs.gamma(S, K, T, sigma) for K in strikes]
ax.plot(strikes, gammas, "b-", lw=2)
ax.axvline(S, color="gray", ls="--", lw=0.8)
ax.set_title(f"Gamma = {o.gamma:+.6f}", fontweight="bold")
ax.set_xlabel("Strike")
ax.grid(True, alpha=0.3)

# Panel 3: Theta vs Strike
ax = axes[0, 2]
thetas_c = [bs.theta(S, K, T, sigma, "call") for K in strikes]
thetas_p = [bs.theta(S, K, T, sigma, "put") for K in strikes]
ax.plot(strikes, thetas_c, "g-", lw=2, label="Call Theta")
ax.plot(strikes, thetas_p, "r-", lw=2, label="Put Theta")
ax.axvline(S, color="gray", ls="--", lw=0.8)
ax.set_title(f"Theta = {o.theta:+.4f}/day", fontweight="bold")
ax.set_xlabel("Strike")
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

# Panel 4: Vega vs Strike
ax = axes[1, 0]
vegas = [bs.vega(S, K, T, sigma) for K in strikes]
ax.plot(strikes, vegas, "m-", lw=2)
ax.axvline(S, color="gray", ls="--", lw=0.8)
ax.set_title(f"Vega = {o.vega:+.4f}", fontweight="bold")
ax.set_xlabel("Strike")
ax.grid(True, alpha=0.3)

# Panel 5: Risk Metrics Radar
ax = axes[1, 1]
metrics = ["Alpha", "Beta", "Sharpe", "Sortino", "Calmar"]
values = [r.alpha * 100, r.beta, r.sharpe_ratio, min(r.sortino_ratio, 5), r.calmar_ratio]
colors_bar = ["green" if v > 0 else "red" for v in values]
bars = ax.barh(metrics, values, color=colors_bar, alpha=0.7, edgecolor="black", linewidth=0.5)
ax.axvline(0, color="black", lw=0.8)
for bar, val in zip(bars, values):
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
            f"{val:+.2f}", va="center", fontsize=8, fontweight="bold")
ax.set_title("Risk Metrics (vs Nifty 50)", fontweight="bold")
ax.grid(True, alpha=0.3, axis="x")

# Panel 6: P&L Profile for recommended strategy
ax = axes[1, 2]
spot_range = np.linspace(S * 0.9, S * 1.1, 100)
legs = o.strategy_legs
pnl = np.zeros_like(spot_range)

for leg in legs:
    K = leg["strike"]
    prem = leg["premium"]
    if leg["type"] == "CE":
        payoff = np.maximum(spot_range - K, 0)
    else:
        payoff = np.maximum(K - spot_range, 0)
    if leg["action"] == "BUY":
        pnl += payoff - prem
    else:
        pnl += prem - payoff

ax.plot(spot_range, pnl, "b-", lw=2)
ax.fill_between(spot_range, pnl, 0, where=(pnl > 0), alpha=0.2, color="green")
ax.fill_between(spot_range, pnl, 0, where=(pnl < 0), alpha=0.2, color="red")
ax.axhline(0, color="black", lw=0.8)
ax.axvline(S, color="gray", ls="--", lw=0.8, label=f"Spot={S:,.0f}")
ax.set_title(f"P&L: {o.recommended_strategy}", fontweight="bold")
ax.set_xlabel("Spot at Expiry")
ax.set_ylabel("P&L (Rs.)")
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(REPORTS_DIR / f"greeks_{TICKER.replace('.', '_')}.png"), dpi=150, bbox_inches="tight")
plt.show()
print(f"Greeks chart saved to reports/greeks_{TICKER.replace('.', '_')}.png")

Greeks chart saved to reports/greeks_RELIANCE_NS.png


In [ ]:
# ============================================================
# Quick Scan: Analyze Multiple Stocks at Once
# ============================================================
# Add any tickers you want to compare
SCAN_TICKERS = ["RELIANCE.NS", "TCS.NS", "INFY.NS", "HDFCBANK.NS", "SBIN.NS"]

scan_results = []
for tk in SCAN_TICKERS:
    try:
        res = analyzer.analyze(tk)
        scan_results.append({
            "Ticker": tk.replace(".NS", ""),
            "Price": f"Rs.{res.technical.price:,.1f}",
            "Signal": res.signal.primary_action,
            "Score": res.signal.confidence_score,
            "RSI": f"{res.technical.rsi_14:.0f}",
            "Trend": res.technical.trend_regime,
            "Vol Regime": res.technical.vol_regime,
            "Beta": f"{res.risk.beta:.2f}",
            "Alpha": f"{res.risk.alpha:.2%}",
            "Sharpe": f"{res.risk.sharpe_ratio:.2f}",
            "Entry": f"Rs.{res.signal.entry_price:,.0f}",
            "Stop": f"Rs.{res.signal.stop_loss:,.0f}",
            "Target": f"Rs.{res.signal.target_2:,.0f}",
            "R:R": f"{res.signal.risk_reward_ratio:.1f}",
            "Strategy": res.options.recommended_strategy,
        })
        print(f"  [OK] {tk}")
    except Exception as e:
        print(f"  [FAIL] {tk}: {e}")

scan_df = pd.DataFrame(scan_results)
print(f"\n{'='*80}")
print(f"  MULTI-STOCK SCAN RESULTS ({len(scan_results)} stocks)")
print(f"{'='*80}")
display(scan_df.style.set_properties(**{"text-align": "center"})
        .set_table_styles([{"selector": "th", "props": [("text-align", "center")]}]))

  496 candles loaded (2024-02-12 to 2026-02-11)
Computing technical indicators...
Generating trading signal...
Analyzing options & Greeks...
Computing risk metrics (vs Nifty 50)...
  [OK] RELIANCE.NS
  496 candles loaded (2024-02-12 to 2026-02-11)
Computing technical indicators...
Generating trading signal...
Analyzing options & Greeks...
Computing risk metrics (vs Nifty 50)...
  [OK] TCS.NS
  496 candles loaded (2024-02-12 to 2026-02-11)
Computing technical indicators...
Generating trading signal...
Analyzing options & Greeks...
Computing risk metrics (vs Nifty 50)...
  [OK] INFY.NS
  496 candles loaded (2024-02-12 to 2026-02-11)
Computing technical indicators...
Generating trading signal...
Analyzing options & Greeks...
Computing risk metrics (vs Nifty 50)...
  [OK] HDFCBANK.NS
  496 candles loaded (2024-02-12 to 2026-02-11)
Computing technical indicators...
Generating trading signal...
Analyzing options & Greeks...
Computing risk metrics (vs Nifty 50)...
  [OK] SBIN.NS

  MULTI-STOC

,Ticker,Price,Signal,Score,RSI,Trend,Vol Regime,Beta,Alpha,Sharpe,Entry,Stop,Target,R:R,Strategy
0,RELIANCE,"Rs.1,468.7",SHORT,40,69,SIDEWAYS,NORMAL,1.19,-7.74%,-0.16,"Rs.1,472","Rs.1,529","Rs.1,379",1.6,Bear Put Spread
1,TCS,"Rs.2,909.8",SHORT,32,34,DOWNTREND,EXTREME,0.90,-22.10%,-0.90,"Rs.2,916","Rs.3,071","Rs.2,668",1.6,Bear Put Spread
2,INFY,"Rs.1,471.9",SHORT,32,19,DOWNTREND,EXTREME,1.17,-11.58%,-0.30,"Rs.1,475","Rs.1,562","Rs.1,337",1.6,Bear Put Spread
3,HDFCBANK,Rs.927.1,STRONG_SELL,20,54,DOWNTREND,LOW,0.85,8.09%,0.62,Rs.929,Rs.963,Rs.873,1.6,Bear Put Spread
4,SBIN,"Rs.1,182.9",BUY,66,75,UPTREND,EXTREME,0.93,21.23%,1.00,"Rs.1,181","Rs.1,114","Rs.1,287",1.6,Bull Call Spread


: 